# AI2002 - Group 5 - Joint Fine-Grained Opinion Extraction and Overall Rating Prediction from Real-World Hotel Reviews

## 0. Cài đặt môi trường (Environment Setup & Auto-install)

Với dự án nghiên cứu về **Review Chatbot for Hotel Question Answering**, hệ thống được thiết lập cơ chế **tự động kiểm tra và cài đặt thư viện** từ file `requirements.txt` vào môi trường ảo (`.venv`):

1. **Tự động nhận diện môi trường ảo**: Cài đặt trực tiếp vào Python Kernel (`sys.executable`) đang chạy trong notebook.
2. **Kiểm tra thông minh**: Chỉ cài đặt các gói còn thiếu từ `requirements.txt`, tránh tốn thời gian khi chạy lại notebook.
3. **Các thư viện chính**:
   - **Pandas / NumPy**: Xử lý dữ liệu bảng quy mô lớn và tính toán đại số ma trận.
   - **PyArrow**: Engine C++ tối ưu hóa nạp dữ liệu siêu tốc và xử lý định dạng Parquet.
   - **SQLite3**: Cơ sở dữ liệu quan hệ cục bộ lưu trữ dữ liệu có cấu trúc.
   - **Matplotlib / Seaborn**: Trực quan hóa dữ liệu EDA và biểu đồ phân tích.
   - **Scikit-learn / NLTK**: Xử lý ngôn ngữ tự nhiên (NLP), TF-IDF và thuật toán Cosine Similarity.
   - **Tqdm / Ipywidgets**: Thanh tiến trình hiển thị trực quan trong quá trình xử lý văn bản.


In [1]:
import os
import re
import sqlite3
import pandas as pd
import numpy as np
import pyarrow as pa
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import nltk

# Tự động tải các gói tài nguyên cần thiết cho NLTK 
for res in ['punkt', 'punkt_tab', 'stopwords']:
    try:
        nltk.download(res, quiet=True)
    except Exception:
        pass

# Hiển thị bảng phiên bản chi tiết các thư viện
print(f"{'Thư viện':<20} {'Phiên bản':>15}")
print("-" * 37)
print(f"{'Python':<20} {os.sys.version.split()[0]:>15}")
print(f"{'Pandas':<20} {pd.__version__:>15}")
print(f"{'PyArrow':<20} {pa.__version__:>15}")
print(f"{'NumPy':<20} {np.__version__:>15}")
print(f"{'Matplotlib':<20} {matplotlib.__version__:>15}")
print(f"{'Seaborn':<20} {sns.__version__:>15}")
print(f"{'Scikit-learn':<20} {sklearn.__version__:>15}")
print(f"{'NLTK':<20} {nltk.__version__:>15}")
print(f"{'SQLite3':<20} {sqlite3.sqlite_version:>15}")
print(f"{'Regex (re)':<20} {re.__version__ if hasattr(re, '__version__') else 'built-in':>15}")

# Khởi tạo thư mục và kết nối SQLite
db_dir = 'Data' if os.path.exists('Data') else 'Data'
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, 'chatbot.db')
conn = sqlite3.connect(db_path)

print(f"\nSQLite đã sẵn sàng tại: {db_path}")
print("--- Môi trường đã được khởi tạo thành công ---")


Thư viện                   Phiên bản
-------------------------------------
Python                        3.13.9
Pandas                         2.3.3
PyArrow                       21.0.0
NumPy                          2.3.5
Matplotlib                    3.10.6
Seaborn                       0.13.2
Scikit-learn                   1.7.2
NLTK                           3.9.2
SQLite3                       3.51.0
Regex (re)                     2.2.1

SQLite đã sẵn sàng tại: Data\chatbot.db
--- Môi trường đã được khởi tạo thành công ---


## 1. Nhập và Kiểm tra Dataset (Dataset Loading & Exploration)

Trong phần này, nhóm áp dụng giải pháp **Tối ưu hóa nạp dữ liệu quy mô lớn** (~740MB, >780.000 dòng):
1. **Cơ chế Caching thông minh (Parquet Optimization)**:
   - Hệ thống tự động kiểm tra định dạng nhị phân dạng cột **Parquet** (`tripadvisor_review_hotel_dataset.parquet`). Nếu chưa tồn tại, chương trình sẽ tự động đọc CSV với engine **PyArrow** đa luồng và tạo cache Parquet (chuẩn nén Snappy).
2. **Kiểm tra cấu trúc và các trường thông tin quan trọng**:
   - **Thông tin cơ sở lưu trú**: `hotel_name`, `hotel_province`, `hotel_address`, `hotel_star`.
   - **Điểm đánh giá và các khía cạnh dịch vụ**: `normalized_score`, `Value`, `Rooms`, `Location`, `Cleanliness`, `Service`, `Sleep_Quality`.
   - **Dữ liệu văn bản phục vụ Chatbot Hỏi Đáp (QA)**: `normalized_title` (tiêu đề đánh giá), `normalized_content` (nội dung đánh giá chi tiết), `Word_count`, `language_code`, `language`.
   - **Bối cảnh chuyến đi & Thời gian**: `trip_type`, `Date`, `month`, `year`.


In [2]:
import os
import time
import pandas as pd

# Tạo thư mục lưu trữ biểu đồ trực quan hóa nếu chưa tồn tại
os.makedirs('Data/charts_img', exist_ok=True)

# Đường dẫn các file dữ liệu
csv_path = 'Data/tripadvisor_review_hotel_dataset.csv'
parquet_path = 'Data/tripadvisor_review_hotel_dataset.parquet'

start_time = time.time()

# Chiến lược nạp dữ liệu tối ưu với Parquet Cache
if os.path.exists(parquet_path):
    print(f"Tìm thấy cache Parquet: {parquet_path}")
    print("Đang nạp dữ liệu từ file Parquet...")
    df = pd.read_parquet(parquet_path)
    load_time = time.time() - start_time
    print(f"Nạp dữ liệu hoàn tất trong: {load_time:.2f} giây")
elif os.path.exists(csv_path):
    print(f"Tìm thấy file dataset CSV: {csv_path}")
    print("Đang đọc CSV và tạo cache Parquet...")
    df = pd.read_csv(csv_path, engine='pyarrow', encoding='utf-8')
    read_time = time.time() - start_time
    print(f"Đọc CSV thành công trong {read_time:.2f} giây")
    
    print("Đang lưu cache Parquet...")
    df.to_parquet(parquet_path, engine='pyarrow', index=False)
    total_time = time.time() - start_time
    print(f"Đã tạo cache Parquet thành công tại: {parquet_path} (Thời gian: {total_time:.2f}s)")
else:
    raise FileNotFoundError(f"Không tìm thấy dataset tại '{csv_path}' hoặc '{parquet_path}'.")

# Thống kê tập dữ liệu
ram_usage_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
active_path = parquet_path if os.path.exists(parquet_path) else csv_path
file_size_mb = os.path.getsize(active_path) / (1024 ** 2)

print(f"\nKích thước tập dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")
print(f"Dung lượng file trên ổ cứng: {file_size_mb:.2f} MB")
print(f"Dung lượng RAM chiếm: {ram_usage_mb:.2f} MB")
print("\nCác cột dữ liệu:")
print(list(df.columns))

# Hiển thị 5 dòng đầu tiên
print("\nXem trước 5 dòng đầu tiên:")
display(df.head())


FileNotFoundError: Không tìm thấy dataset tại 'Data/tripadvisor_review_hotel_dataset.csv' hoặc 'Data/tripadvisor_review_hotel_dataset.parquet'.

## 2. Data Analysis with SQL & Python   

### 2.1. Đưa dữ liệu từ DataFrame vào bảng SQLite để truy vấn.  

### 2.2. SQL Queries để giải quyết 3 RQ.

Dự án bao gồm 5 Research Questions bao gồm:

RQ1: CRAM-ABSA có cải thiện joint ABSA + rating prediction so với các baseline single-task/multi-task đơn giản trên cùng Hotel-Disjoint split hay không?

***Does CRAM-ABSA improve joint ABSA and rating prediction compared to simple single-task/multi-task baselines under the same Hotel-Disjoint split?***

RQ2: Continuous Rating Head có giúp biểu diễn toàn cục và cải thiện extraction hay chỉ cải thiện riêng rating?

***Does the Continuous Rating Head facilitate global representation and improve extraction, or does it solely enhance rating prediction?***

RQ3: Conflict Attenuation Gating có thực sự giúp các review mixed/conflicting sentiment hay không?

***Does the Conflict Attenuation Gating mechanism genuinely assist with reviews containing mixed or conflicting sentiments?***

RQ4: Tại sao Overlap F1 cao nhưng Exact F1 thấp, và lỗi chủ yếu nằm ở boundary, aspect mapping hay sentiment?

***Why is the Overlap F1 score high while the Exact F1 score is low, and do the errors primarily stem from boundaries, aspect mapping, or sentiment?***

RQ5: Hotel-Disjoint evaluation thay đổi hiệu năng thế nào so với random review split?

***How does performance under the Hotel-Disjoint evaluation differ from that of a random review split?***